<a href="https://colab.research.google.com/github/rafaqatkhattak5/connected-karak-news-classifier/blob/main/Train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets accelerate scikit-learn pandas

In [2]:
# Load your data

import pandas as pd
from google.colab import files

files.upload()   # choose news_data.csv from your phone
df = pd.read_csv("news_data.csv")

print(len(df), df["label"].nunique())
print(df["label"].value_counts())

Saving news_data.csv to news_data.csv
86 7
label
Other        20
Education    11
Politics     11
Crime        11
Health       11
Sports       11
Weather      11
Name: count, dtype: int64


In [3]:
# Labels and split

from sklearn.model_selection import train_test_split

labels = sorted(df["label"].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
df["label_id"] = df["label"].map(label2id)

train_df, test_df = train_test_split(
    df, test_size=0.25, random_state=42, stratify=df["label_id"]
)

print(label2id)
print(len(train_df), len(test_df))

{'Crime': 0, 'Education': 1, 'Health': 2, 'Other': 3, 'Politics': 4, 'Sports': 5, 'Weather': 6}
64 22


In [4]:
#Tokenize

from transformers import AutoTokenizer
from datasets import Dataset

model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True,
                     padding="max_length", max_length=64)

def to_dataset(d):
    d = d[["text", "label_id"]].rename(columns={"label_id": "labels"})
    return Dataset.from_pandas(d, preserve_index=False).map(tokenize, batched=True)

train_ds = to_dataset(train_df)
test_ds = to_dataset(test_df)

print(len(train_ds), len(test_ds))

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

Map:   0%|          | 0/22 [00:00<?, ? examples/s]

64 22


In [5]:
# Train

from transformers import (AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, set_seed)
import numpy as np

set_seed(42)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = np.argmax(logits, axis=-1)
    return {"accuracy": (y_pred == y_true).mean()}

args = TrainingArguments(
    output_dir="news_model",
    num_train_epochs=20,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-5,
    warmup_steps=10,
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="no",
    fp16=True,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_

Epoch,Training Loss,Validation Loss,Accuracy
1,1.969715,1.979887,0.136364
2,2.005880,1.966576,0.136364
3,1.947644,1.949329,0.136364
4,1.935793,1.922048,0.272727
5,1.889587,1.684965,0.272727
6,1.658275,1.477236,0.363636
7,1.461301,1.260939,0.772727
8,1.200490,1.128654,0.772727
9,1.030998,0.952275,0.818182
10,0.787105,0.761547,0.818182


TrainOutput(global_step=160, training_loss=0.9505630299448967, metrics={'train_runtime': 1292.0242, 'train_samples_per_second': 0.991, 'train_steps_per_second': 0.124, 'total_flos': 42099658752000.0, 'train_loss': 0.9505630299448967, 'epoch': 20.0})

In [6]:
# Evaluate

import numpy as np
from sklearn.metrics import classification_report

out = trainer.predict(test_ds)
pred = np.argmax(out.predictions, axis=-1)
true = out.label_ids

for t, y, p in zip(test_df["text"], true, pred):
    if y != p:
        print(f"TRUE: {id2label[y]:10} PRED: {id2label[p]:10} | {t}")

print()
print(classification_report(true, pred,
      labels=list(range(len(labels))), target_names=labels,
      zero_division=0))

TRUE: Other      PRED: Education  | Job opportunities announced at local company
TRUE: Health     PRED: Other      | ڈینگی کے کیسز میں اضافہ
TRUE: Weather    PRED: Other      | په کرک کې د سړې هوا څپه راغلې
TRUE: Other      PRED: Weather    | Electricity load shedding continues in several areas

              precision    recall  f1-score   support

       Crime       1.00      1.00      1.00         2
   Education       0.75      1.00      0.86         3
      Health       1.00      0.67      0.80         3
       Other       0.60      0.60      0.60         5
    Politics       1.00      1.00      1.00         3
      Sports       1.00      1.00      1.00         3
     Weather       0.67      0.67      0.67         3

    accuracy                           0.82        22
   macro avg       0.86      0.85      0.85        22
weighted avg       0.83      0.82      0.82        22



In [7]:
# prediction function

import torch

def predict(texts, threshold=0.5):
    model.eval()
    inputs = tokenizer(texts, truncation=True, padding=True,
                       max_length=64, return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)
    for text, p in zip(texts, probs):
        i = p.argmax().item()
        label = id2label[i] if p[i] >= threshold else "Other (unsure)"
        print(f"{label:15} {p[i]:.0%}  |  {text}")

predict([
    "پولیس نے چور کو پکڑ لیا",
    "کل بارش کا امکان ہے",
    "د کرک ځوانانو کرکټ ګټلی",
    "Vegetable prices go up before Eid",
    "Heavy snowfall expected in Karak tonight",
])

Crime           95%  |  پولیس نے چور کو پکڑ لیا
Weather         95%  |  کل بارش کا امکان ہے
Sports          94%  |  د کرک ځوانانو کرکټ ګټلی
Other           95%  |  Vegetable prices go up before Eid
Weather         95%  |  Heavy snowfall expected in Karak tonight


In [8]:
# Saved the model

from google.colab import drive
drive.mount("/content/drive")

save_path = "/content/drive/MyDrive/news_classifier_v2"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

from transformers import pipeline
clf = pipeline("text-classification", model=save_path, tokenizer=save_path)
print(clf("Prices of sugar go up in the market"))
print(clf.model.config.id2label)

Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': 'Other', 'score': 0.9593607187271118}]
{0: 'Crime', 1: 'Education', 2: 'Health', 3: 'Other', 4: 'Politics', 5: 'Sports', 6: 'Weather'}
